# Assessment 2: Pemodelan Machine Learning (Klasifikasi & Klastering) — Revisi

**Dataset:** New York City Airbnb 2019

Notebook ini berisi implementasi lengkap dari pipa pemrosesan data (preprocessing pipeline), pemodelan Supervised Learning (Klasifikasi Popularitas), dan Unsupervised Learning (Klastering Geografis & Ekonomi) dengan validasi silang, penyetelan ambang keputusan, dan pemilihan konfigurasi klastering secara objektif.


In [ ]:
%pip install scikit-learn
import warnings
warnings.filterwarnings("ignore")

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


from IPython.display import Markdown, display

RANDOM_STATE = 42

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)

def section(title):
    display(Markdown(f"## {title}"))

def ask_plot(question):
    display(Markdown(f"**Pertanyaan grafik:** {question}"))

def explain_plot(points):
    display(Markdown(f"**Interpretasi:**\n" + "\n".join(f"- {p}" for p in points)))

def fmt_num(x, digits=2):
    if pd.isna(x):
        return "NA"
    if abs(x) >= 1000:
        return f"{x:,.{digits}f}"
    return f"{x:.{digits}f}"


In [2]:
# Unduh / muat dataset dengan fallback ke file lokal
selected_cols = [
    "id", "host_id",
    "neighbourhood_group", "neighbourhood",
    "latitude", "longitude",
    "room_type",
    "price", "minimum_nights",
    "number_of_reviews", "last_review", "reviews_per_month",
    "calculated_host_listings_count", "availability_365",
]

try:
    path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
    csv_path = f"{path}/AB_NYC_2019.csv"
    print(f"[kagglehub] Dataset diunduh: {path}")
except Exception:
    csv_path = "AB_NYC_2019.csv"
    print(f"[local] Menggunakan file lokal: {csv_path}")

df_raw = pd.read_csv(csv_path, usecols=selected_cols, parse_dates=["last_review"])

# Filter data tidak valid
mask_valid = (
    (df_raw["price"] > 0)
    & (df_raw["availability_365"].between(0, 365))
    & (df_raw["latitude"].between(40.45, 40.95))
    & (df_raw["longitude"].between(-74.30, -73.65))
)
df = df_raw.loc[mask_valid].copy()

print(f"Baris setelah pembersihan: {len(df):,}")
print(f"Baris terbuang: {len(df_raw) - len(df):,}")


[kagglehub] Dataset diunduh: C:\Users\erlanggadewa\.cache\kagglehub\datasets\dgomonov\new-york-city-airbnb-open-data\versions\3
Baris setelah pembersihan: 48,884
Baris terbuang: 11


In [5]:
df.head()

,id,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,...,calculated_host_listings_count,availability_365,log_price,log_min_nights,last_review_age_days,has_reviews,distance_to_center_km,availability_365_capped,minimum_nights_capped,log_number_of_reviews
0,2539,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,...,6,365,5.010635,0.693147,262.0,1,12.337898,365,1,2.302585
1,2595,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,...,2,355,5.420535,0.693147,48.0,1,0.508366,355,1,3.828641
2,3647,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,...,1,365,5.017280,1.386294,1825.0,0,6.757240,365,3,0.000000
3,3831,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,...,1,194,4.499810,0.693147,3.0,1,8.387034,194,1,5.602119
4,5022,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,...,1,0,4.394449,2.397895,231.0,1,5.701496,0,10,2.302585
